# pawpaw

Compile a sentence of English into a small LoRA adapter that runs locally and
returns a one-word answer in milliseconds. Useful for the routing decisions an
agent makes a thousand times a day:

- *Is this a trivial question or something substantive?*
- *Is the user frustrated?*
- *Is this about code, writing, or math?*

Each one becomes a `.paw` file you load once and call many times. Several of
them in the same process share a base model, so you pay the GPU memory cost
once.

The whole API is two functions:

```python
pawpaw.compile(spec, save_to=...)  # slow: synthesize data, train LoRA, package
program = pawpaw.load("...")        # fast: read bundle, hot-attach adapter
program("user message")             # very fast: classify
```

This notebook walks through:

1. **Compile** a program from a spec + a few seed examples (offline).
2. **Load + call** it.
3. **Multi-program agent** — three classifiers in one process, sharing a base model.
4. **Long prompts** — the runtime targets 100s of tokens of user input, not just headlines.
5. **Speed comparison** vs the upstream `programasweights` runtime, on the same `.paw` bundles.
6. **Cross-runtime** — the `.paw` v2 format is binary-compatible, so we can also run upstream programs locally.

Hardware: detected automatically — Metal on macOS, CUDA on Linux GPUs, CPU otherwise.


## 1. Compile a program from a spec

`pawpaw.compile(spec, save_to=..., examples=...)` is the slow step. It
synthesizes a training set from the spec (or uses your seed examples), trains
a LoRA adapter against a small base model (`Qwen3-0.6B` by default), converts
the adapter to an f16 GGUF file, and writes a portable bundle to disk. You only do
this once per program.

In production you'd usually pass `llm_model_path="…/qwen3-4b-instruct.gguf"`
and let the synth stage generate hundreds of diverse examples. To keep this
notebook self-contained we instead pass ~100 seed (input, output) pairs.

In [ ]:
import pawpaw

TRIAGE_EXAMPLES = [
    ("How are you doing today?",                                    "trivial"),
    ("What time is it in Tokyo?",                                   "trivial"),
    ("Can you say hi?",                                             "trivial"),
    ("Hello!",                                                      "trivial"),
    ("Why is my React component re-rendering on every keypress?",   "substantive"),
    ("Explain how attention heads work in transformers.",           "substantive"),
    ("Help me debug this null pointer in Java.",                    "substantive"),
    ("Write a proof that sqrt(2) is irrational.",                   "substantive"),
] * 12  # repeat to reach min_examples; in real use you'd synthesize them

result = pawpaw.compile(
    spec="Decide if the user message is 'trivial' (small talk / one-line lookup) "
         "or 'substantive' (needs reasoning or domain knowledge). "
         "Output exactly one word.",
    save_to="programs/triage",
    examples=TRIAGE_EXAMPLES,
    rank=4,
    epochs=1,
)

print(f"saved {result.paw_path} ({result.paw_path.stat().st_size / 1024 / 1024:.1f} MB)")
print(f"bundle dir: {result.bundle_dir}")


In [ ]:
# What's in the bundle?
import os
for f in sorted(os.listdir(result.bundle_dir)):
    p = result.bundle_dir / f
    print(f"  {f:24}  {p.stat().st_size / 1024:7.1f} KB")


## 2. Load and call the program

`pawpaw.load(path)` returns a `Program` you call like a function. The first
call lazily loads a Llama context for the base model — that takes a couple of
seconds. Subsequent calls reuse the warm prefix KV cache.

In [ ]:
triage = pawpaw.load("programs/triage")

print(triage("How's the weather in San Francisco?"))
print(triage("How do I configure GitHub Actions to cache npm dependencies across workflow runs?"))


## 3. Many programs in one agent process

Real agents need several routing decisions per turn. Loading three programs
that share a base model costs roughly the same memory as loading one — pawpaw
caches the Llama context behind the scenes, keyed on `(base_model, n_ctx,
n_gpu_layers)`. Each `Program` brings its own LoRA adapter (a few MB) and
prefix KV cache.

Switching between programs on consecutive calls swaps the adapter + restores
the prefix KV in single-digit milliseconds.

In [ ]:
# Compile two more programs — same shape, different specs.

MOOD_EXAMPLES = [
    ("Why doesn't this stupid thing work??",   "frustrated"),
    ("This is the third time I've reported this bug!", "frustrated"),
    ("Just give me a working example, please.","frustrated"),
    ("I'm going in circles, this is wasting my afternoon.", "frustrated"),
    ("Sounds great, thanks!",                  "calm"),
    ("Got it, that worked perfectly.",         "calm"),
    ("Could you help me understand this?",     "calm"),
    ("Ok, what should I try next?",            "calm"),
] * 12

DOMAIN_EXAMPLES = [
    ("How do I sort a list in Python without losing the original order?", "code"),
    ("Why does my regex match more than I expected?", "code"),
    ("Help me debug this Rust borrow checker error.", "code"),
    ("What's the cleanest way to do retries with exponential backoff?", "code"),
    ("Help me rewrite this paragraph to sound more concise.", "writing"),
    ("Is 'who' or 'whom' correct in this sentence?", "writing"),
    ("Suggest a stronger opening line for my essay about cities.", "writing"),
    ("Edit this email to be more polite but firm.", "writing"),
] * 12

pawpaw.compile(
    spec="Classify the user message as 'frustrated' or 'calm'. Output one word.",
    save_to="programs/mood", examples=MOOD_EXAMPLES, rank=4, epochs=1,
)
pawpaw.compile(
    spec="Classify the user message as 'code' (programming question) or 'writing' "
         "(natural-language editing). Output one word.",
    save_to="programs/domain", examples=DOMAIN_EXAMPLES, rank=4, epochs=1,
)


In [ ]:
mood   = pawpaw.load("programs/mood")
domain = pawpaw.load("programs/domain")
# triage was loaded earlier and stays warm; mood + domain reuse its Llama context.

def route(message: str) -> dict:
    return {
        "triage": triage(message),
        "mood":   mood(message),
        "domain": domain(message),
    }

for msg in [
    "Hey, can you suggest a better word than 'utilize' here?",
    "Why does my regex match three times instead of once??",
    "How are you?",
    "I'M REALLY FRUSTRATED that you keep giving the same wrong answer!!",
]:
    print(f"{route(msg)}   ←   {msg!r}")


## 4. Long prompts

The default context window is 4096 tokens. The runtime caches the prompt
prefix (the spec + few-shot examples) on the GPU after the first call, so the
per-call cost grows only with the *user input* length. A few hundred tokens
classify in tens of milliseconds on modern GPUs.

In [ ]:
import time

LONG_INPUT = '''
Hi, I'm working on the data ingestion pipeline for our analytics platform. We
have about 400 GB of nightly events landing in S3 in newline-delimited JSON,
partitioned by date. I'm using a Spark job on EMR to deduplicate, normalize the
event schemas (we have ~30 event types with subtle differences across versions),
and write Parquet to a downstream bucket that Athena reads. The job has been
running fine for months but last week I started seeing intermittent OOMs on the
shuffle stage — only on Tuesdays, weirdly enough, which is when our marketing
team runs an A/B test export that roughly triples one event type's volume. I've
tried bumping executor memory and reducing partition size but the failure mode
isn't really getting better. Can you help me think through what's going on?
'''.strip()

n = 30
start = time.perf_counter()
for _ in range(n):
    label = triage(LONG_INPUT)
elapsed = time.perf_counter() - start

print(f"label: {label}")
print(f"per-call: {1000 * elapsed / n:.1f} ms ({n} iterations on a ~400-token prompt)")


## 5. Speed comparison vs upstream `programasweights`

The `.paw` v2 binary format is shared with upstream `programasweights`, so we
can download an upstream-compiled program and run it through *our* runtime for
an apples-to-apples comparison.

In [ ]:
try:
    import programasweights as paw

    upstream_program_id = "3894309a9eac5c584899"  # public sentiment classifier
    print(f"Downloading upstream bundle ({upstream_program_id})...")
    handle = paw.function(upstream_program_id, n_gpu_layers=0, n_ctx=512)
    upstream_bundle_dir = handle._program_dir
    del handle  # release upstream's llama context before we open our own

    upstream_local = pawpaw.load(upstream_bundle_dir)
    test_text = "The service was decent, but the prices were a bit high for the portion sizes."
    upstream_local(test_text)  # warmup

    n = 30
    s = time.perf_counter()
    for _ in range(n):
        upstream_local(test_text)
    upstream_t = (time.perf_counter() - s) / n

    s = time.perf_counter()
    for _ in range(n):
        triage(test_text)
    pawpaw_t = (time.perf_counter() - s) / n

    print()
    print("| program        | per-call (ms) |")
    print("|----------------|--------------:|")
    print(f"| pawpaw triage  | {pawpaw_t * 1000:>5.1f}         |")
    print(f"| upstream sst-2 | {upstream_t * 1000:>5.1f}         |")
except Exception as e:
    print(f"Skipped (upstream programasweights not available or no network): {e}")


## 6. Hardware detection + cross-runtime use

`pawpaw.load(..., verbose=True)` shows what the underlying llama.cpp picked.
On macOS you should see Metal init logs; on a CUDA Linux box you'd see CUDA
device info; otherwise it falls back to CPU.

A pawpaw `.paw` is binary-compatible with upstream `programasweights`, so the
program we compiled in cell 1 also runs in upstream's runtime, and vice
versa. That makes it easy to mix and match (and benchmark).

In [ ]:
import platform
print(f"Host: {platform.system()} / {platform.machine()} / Python {platform.python_version()}")

# Show how to ask the runtime to be loud (one-time, just for visibility).
fresh = pawpaw.load("programs/triage", verbose=True)
print(fresh("Quick check: is this trivial?"))
del fresh


## 7. Where to go next

- **Real synthesis:** drop `examples=` and pass
  `llm_model_path="…/qwen3-4b-instruct-q4_k_m.gguf"`. The synth stage will
  generate hundreds of varied training examples, including adversarial inputs
  and length/format variations.
- **Bigger LoRA for harder tasks:** `rank=16, epochs=3` is a good production
  default. Bump further only when val accuracy is plateaued.
- **Longer context:** `pawpaw.load("…", n_ctx=8192)` if your inputs are
  routinely > 1000 tokens. The base model goes up to 40k.
- **Benchmarks:** `python -m benchmarks.sst2_accuracy compile/eval` and
  `python -m benchmarks.speed` for rigorous measurements vs upstream.
- **Distribution:** `programs/triage.paw` is a single file — copy it between
  machines, ship it with your service, hash-pin it in CI.